In [ ]:
# ======================================================================
# CLEAN DUBLIN RENT: RTB/ESRI rent index for Dublin, by district
# ======================================================================
# Input:  just_dublin_dataset\just_dublin_processed_dataset\rent\rent_dublin.csv
# Output: just_dublin_dataset\Just_dublin_clean_dataset\rent\rent_dublin_clean.csv
#         just_dublin_dataset\Just_dublin_clean_dataset\rent\rent_district_headline.csv
#
# What this is for (Student 3):
#   - temporal analysis of rents and prices by Dublin district
#   - context for the house price model (rent per district per quarter)
#
# These are published averages, not single tenancies, so there is little
# to clean. Mainly: checks, and a small headline table with one rent per
# district per quarter (the average of the local areas in each district,
# because whole-district rows stop after 2021).
#
# Needs the Dublin processed notebooks (00 and 04) to be run first.
# ======================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

pd.set_option('display.max_columns', None)

base = r'C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project'

rent_file    = base + r'\just_dublin_dataset\just_dublin_processed_dataset\rent\rent_dublin.csv'
dublin_clean = base + r'\just_dublin_dataset\Just_dublin_clean_dataset'

In [ ]:
rent = pd.read_csv(rent_file)

print(rent.shape)
rent.head()

In [ ]:
# Quick checks: spaces in text, missing values, duplicates

for col in ['quarter', 'bedrooms', 'property_type', 'location', 'location_level']:
    rent[col] = rent[col].str.strip()

print('Missing values:')
print(rent.isnull().sum())
print()
key = ['quarter', 'location', 'bedrooms', 'property_type']
print('Duplicate rows:', rent.duplicated(subset=key).sum())

In [ ]:
# The only missing values should be dublin_district / district_id on the
# whole-county 'Dublin' rows (they are not one district), and local_authority
# (not in the rent index).

print(rent[rent['dublin_district'].isnull()]['location_level'].value_counts())

In [ ]:
# Checking the rent values look sensible (monthly rent in euro)

print(rent['avg_rent'].describe().round(0))
print('Rents below 100 or above 10,000:', ((rent['avg_rent'] < 100) | (rent['avg_rent'] > 10000)).sum())

In [ ]:
# Which quarters have a whole-district row (location = 'Dublin 12')?
# The rent index stopped publishing these after 2021, but it still publishes
# the smaller areas inside each district ('Crumlin, Dublin 12') every quarter.

district_rows = rent[rent['location_level'] == 'district']
print('Whole-district rows per quarter:')
print(district_rows.groupby('quarter').size())

local_rows = rent[(rent['location_level'] == 'local') & (rent['dublin_district'] != 'Dublin (no district)')]
print()
print('Quarters with local-area rows, per district (min / max):',
      local_rows.groupby('dublin_district')['quarter'].nunique().min(), '/',
      local_rows.groupby('dublin_district')['quarter'].nunique().max())

In [ ]:
# Flag for the 'all homes' rows: all bedrooms and all property types.
# These are the rows used for one average rent per area.

rent['is_all_homes'] = 0
rent.loc[(rent['bedrooms'] == 'All bedrooms') &
         (rent['property_type'] == 'All property types'), 'is_all_homes'] = 1

print('All-homes rows:', rent['is_all_homes'].sum())

In [ ]:
# District headline rent = the average of the local areas in that district,
# for every district and quarter (same method for all, so they can be
# compared). The number of local areas used is kept as n_local_areas.
# This is a simple average of area averages, because the rent index does
# not give the number of tenancies behind each figure.

local_all = rent[(rent['location_level'] == 'local') &
                 (rent['is_all_homes'] == 1) &
                 (rent['dublin_district'] != 'Dublin (no district)')]

headline = local_all.groupby(['dublin_district', 'district_id', 'quarter', 'year', 'quarter_no'])['avg_rent'].agg(['mean', 'count']).reset_index()
headline = headline.rename(columns={'mean': 'avg_rent', 'count': 'n_local_areas'})
headline['avg_rent'] = headline['avg_rent'].round(2)
headline = headline.sort_values(['district_id', 'quarter']).reset_index(drop=True)

print(headline.shape)
headline.head()

In [ ]:
# Every district should have all 20 quarters (2021Q1 to 2025Q4)

quarters_per_district = headline.groupby('dublin_district')['quarter'].count()
print('Districts:', headline['dublin_district'].nunique())
print('Quarters per district (min / max):', quarters_per_district.min(), '/', quarters_per_district.max())
print()
print('Local areas behind each district (average):')
print(headline.groupby('dublin_district')['n_local_areas'].mean().round(1).sort_values())

In [ ]:
# Sense check for 2021: where the whole-district row still exists, it should
# be close to the average of its local areas.

check = district_rows[(district_rows['bedrooms'] == 'All bedrooms') &
                      (district_rows['property_type'] == 'All property types')]
check = check[['dublin_district', 'quarter', 'avg_rent']].rename(columns={'avg_rent': 'district_row'})
check = check.merge(headline[['dublin_district', 'quarter', 'avg_rent']], on=['dublin_district', 'quarter'])
check['difference_%'] = ((check['avg_rent'] - check['district_row']) / check['district_row'] * 100).round(1)

print('Average difference:', check['difference_%'].abs().mean().round(1), '%')
check.head(10)

In [ ]:
# Adding the higher location levels to the headline table

headline['county'] = 'Dublin'
headline['county_id'] = rent['county_id'].values[0]
headline['local_authority'] = np.nan   # the rent index doesn't use councils for Dublin
headline['province'] = 'Leinster'
headline['nation'] = 'Ireland'

headline.head()

In [ ]:
# Rent change per district from 2021Q1 to 2025Q4, as a quick sense check

first = headline[headline['quarter'] == '2021Q1'].set_index('dublin_district')['avg_rent']
last = headline[headline['quarter'] == '2025Q4'].set_index('dublin_district')['avg_rent']

change = ((last - first) / first * 100).round(1).sort_values()

plt.figure(figsize=(10, 7))
change.plot(kind='barh')
plt.xlabel('% change in average rent, 2021Q1 to 2025Q4')
plt.title('Rent growth by Dublin district')
plt.show()

In [ ]:
# Column order for the full clean table: location first, then details, then rent

rent_clean = rent[['nation', 'province', 'county', 'county_id', 'local_authority',
                   'dublin_district', 'district_id', 'location', 'location_level',
                   'quarter', 'year', 'quarter_no', 'bedrooms', 'property_type',
                   'is_all_homes', 'avg_rent']]

print(rent_clean.shape)
rent_clean.info()

In [ ]:
# Saving both tables.

save_folder = dublin_clean + r'\rent'
os.makedirs(save_folder, exist_ok=True)

rent_clean.to_csv(save_folder + r'\rent_dublin_clean.csv', index=False)
headline.to_csv(save_folder + r'\rent_district_headline.csv', index=False)

print('Saved', rent_clean.shape, 'to', save_folder + r'\rent_dublin_clean.csv')
print('Saved', headline.shape, 'to', save_folder + r'\rent_district_headline.csv')